# The Inverted Index and Safe Dynamic Pruning (WAND, BlockMax-WAND)

This notebook is the *narrative* pillar: it imports the canonical reference implementation in `inverted_index_dynamic_pruning.py` — which **owns every number** — and walks the topic section by section. Each claim is an `assert` in the `.py`; here we run those checks and print the numbers the page and the interactive visualizer mirror.

Run end to end with:

```bash
uv run --with numpy --with jupyter \
    jupyter execute notebooks/inverted-index-dynamic-pruning/01_inverted_index_dynamic_pruning.ipynb
```

In [ ]:
import sys, pathlib
_cands = [pathlib.Path.cwd(),
          pathlib.Path.cwd() / "notebooks" / "inverted-index-dynamic-pruning",
          pathlib.Path("notebooks/inverted-index-dynamic-pruning")]
for _d in _cands:
    if (_d / "inverted_index_dynamic_pruning.py").exists():
        sys.path.insert(0, str(_d)); break
import inverted_index_dynamic_pruning as ii
index = ii.build_inverted_index(ii._WORKED_CORPUS)
print("loaded reference implementation; query =", ii._QUERY)

## 1. The inverted index

The inverted index maps each term to its **postings list** — the documents containing it, sorted by document id, each with the term frequency. BM25 is scored over this structure, and the same per-term upper bounds the postings expose are what make safe pruning possible. We use BM25 verbatim from the published topic (idf "bm25", $k_1=1.5$, $b=0.75$).

In [ ]:
ii.viz_constants()

## 2. Exhaustive document-at-a-time scoring

The baseline is a document-at-a-time (DAAT) scan: accumulate every query term's contribution into every document it touches, then sort. It is exact by construction and gives the ground truth the pruned methods must match. On the shared finance corpus it reproduces the published BM25 ranking.

In [ ]:
ii.worked_ranking()
ii.test_bm25_matches_published_ranking()

## 3. WAND and the safety theorem

WAND keeps, for each query term, a global upper bound $\mathrm{UB}_t$ (its largest possible contribution). With a running threshold $\theta$ (the $k$-th best score so far), it finds the **pivot** — the first document whose cumulative upper bound can reach $\theta$ — and skips everything before it. Because every true score is at most the sum of upper bounds, no document that belongs in the top-$k$ is ever skipped: WAND returns the **exact** top-$k$.

In [ ]:
ii.test_wand_bmw_safety()

## 4. Safety on random instances, and a monotone threshold

The skip is valid only because the threshold never decreases. We verify both the exact-top-$k$ guarantee and threshold monotonicity across many random corpora and values of $k$ — ties at the $k$-th boundary are allowed (a safe method reproduces the same *scores*, even if it breaks a tie differently).

In [ ]:
ii.test_safety_random_strict()

## 5. BlockMax-WAND: tighter local bounds

A single global $\mathrm{UB}_t$ is loose: most of a long postings list scores far below its maximum. BlockMax-WAND stores a per-**block** maximum and uses it as a tighter local upper bound at the pivot, so it skips the expensive full scoring of documents WAND would evaluate. It is still exact, and at scale it scores a small fraction of the collection.

In [ ]:
ii.test_pruning_reduces_full_evals()
ii.pruning_demo()

## 6. No free lunch: the adversarial case

Dynamic pruning is exact but has **no asymptotic guarantee**. When every document scores the same — a flat single-term query — the threshold never rises enough for a skip to fire, and WAND degrades to the full exhaustive scan. The win is a data-dependent constant factor, not a better complexity class.

In [ ]:
ii.test_adversarial_flat_scores_prune_nothing()
ii.test_edge_cases()

---

**Three pillars, one set of numbers.** The rigorous math (the page), the interactive pruning visualizer, and this notebook all read from `inverted_index_dynamic_pruning.py`. Change a number there and the viz and the page must change with it.